# 以 ollama 執行 llama3

# 設定必要的資源包和函式

In [1]:
# %pip install ollama
# %pip install langchain chromadb pypdf fastembed
# %pip install aiofiles

In [2]:
import os
import textwrap
from IPython.display import display
from IPython.display import Markdown
from datetime import datetime
import numpy_financial as npf
import math
import pandas as pd
import numpy as np
import json

import ollama
from ollama import generate
from langchain_community.vectorstores import Chroma
from langchain_community.chat_models import ChatOllama
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain.schema.output_parser import StrOutputParser
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema.runnable import RunnablePassthrough
from langchain.prompts import PromptTemplate
from langchain.vectorstores.utils import filter_complex_metadata

/Users/yanyifu/Documents/_Coding/LLM Predict Stock/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 用來讀取文字檔，轉成embedding

In [3]:
from typing import AsyncIterator, Iterator

from langchain_core.document_loaders import BaseLoader
from langchain_core.documents import Document


class CustomDocumentLoader(BaseLoader):
    """An example document loader that reads a file line by line."""

    def __init__(self, file_path: str) -> None:
        """Initialize the loader with a file path.

        Args:
            file_path: The path to the file to load.
        """
        self.file_path = file_path

    def lazy_load(self) -> Iterator[Document]:  # <-- Does not take any arguments
        """A lazy loader that reads a file line by line.

        When you're implementing lazy load methods, you should use a generator
        to yield documents one by one.
        """
        with open(self.file_path, encoding="utf-8") as f:
            line_number = 0
            for line in f:
                yield Document(
                    page_content=line,
                    metadata={"line_number": line_number, "source": self.file_path},
                )
                line_number += 1

    # alazy_load is OPTIONAL.
    # If you leave out the implementation, a default implementation which delegates to lazy_load will be used!
    async def alazy_load(
        self,
    ) -> AsyncIterator[Document]:  # <-- Does not take any arguments
        """An async lazy loader that reads a file line by line."""
        # Requires aiofiles
        # Install with `pip install aiofiles`
        # https://github.com/Tinche/aiofiles
        import aiofiles

        async with aiofiles.open(self.file_path, encoding="utf-8") as f:
            line_number = 0
            async for line in f:
                yield Document(
                    page_content=line,
                    metadata={"line_number": line_number, "source": self.file_path},
                )
                line_number += 1

## 設定模型參數

In [4]:
model = 'llama3'
temperature = 1
top_p = 0.9
seed = 0
system_prompt = '''你是一個厲害的股市分析師，當正面時回答#非常好或#好，負面時回答#不好或#非常不好，其他則是#無關'''
chunk_size=128
chunk_overlap=10


# f'''<|begin_of_text|><|start_header_id|>system<|end_header_id|> 

#     {system_prompt}<|eot_id|> 
#     <|start_header_id|>user<|end_header_id|>

#     {user_message}<|eot_id|>
#     <|start_header_id|>assistant<|end_header_id|>'''

In [5]:
class RAG:
    vector_store = None
    retriever = None
    chain = None

    def __init__(self):
        self.model = ChatOllama(model=model, max_tokens=128, temperature=temperature, top_p=top_p)
        self.text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
        self.prompt = PromptTemplate.from_template(
            """
            <s> [INST] 你是一個厲害的股市分析師，當正面時回答#非常好或#好，負面時回答#不好或#非常不好，其他則是#無關 [/INST] </s> 
            [INST] 問題: {question} 
            文章: {context} 
            回答: [/INST]
            """
        )

    def ingest(self, pdf_file_path: str):
        docs = CustomDocumentLoader(file_path=pdf_file_path).load()
        chunks = self.text_splitter.split_documents(docs)
        chunks = filter_complex_metadata(chunks)
        # chunks.append(chunks)

        vector_store = Chroma.from_documents(documents=chunks, embedding=FastEmbedEmbeddings())
        self.retriever = vector_store.as_retriever(
            search_type="similarity_score_threshold",
            search_kwargs={
                "k": 3,
                "score_threshold": 0.5,
            },
        )

        self.chain = ({"context": self.retriever, "question": RunnablePassthrough()}
                      | self.prompt
                      | self.model
                      | StrOutputParser())

    def ask(self, query: str):
        if not self.chain:
            return "Please, add a document first."

        return self.chain.invoke(query)

    def clear(self):
        self.vector_store = None
        self.retriever = None
        self.chain = None


In [6]:
# import os

# rag = RAG()
# rag.clear()
# rag.ingest("stock_news/3023/3023_news/20050129")

# response = rag.ask("請根據信邦 (3023)的商業創新計畫的全面性分析，重點關注其對目標市場的預期影響，並強調由投資者樂觀的長期成長預期所推動的股價持續飙升現象進行判斷")   # Please, add a PDF document first.

# print(response)

In [7]:
def lm_response(ragfile, instruction):
    text = ''
    try:
        rag = RAG()
        rag.clear()
        rag.ingest(ragfile)
        text = rag.ask(instruction)
        print(text)
    except Exception as e:
        print(e)
        pass
    # print(text)

    if text.find('非常不好') > 0:
        score = -2
    elif text.find('非常好') > 0:
        score = 2
    elif text.find('不好') > 0:
        score = -1
    elif text.find('好') > 0:
        score = 1
    else:
        score = 0

    return score  # 分數

In [8]:
# content = '''在半導體供應鏈中，IC測試業雖屬後段服務，卻是不可或缺的一環，隨著全球IDM大廠將測試產能委外的發展趨勢逐漸明顯，測試業技術挑戰性越來越高，發展空間也加大。  1顆IC的形成，包括設計、生產製造、封裝到測試，在半導體供應鏈分工過程中，測試是最後階段的工程服務，但在結合前段設計、生產製造及後段封裝的一貫化服務策略下，測試業逐漸發展成為具競爭力的分工夥伴。研發工程師需求增多研發工程師需求增多測試業做些什麼呢？1位測試廠處長表示，測試主要在檢測IC的功能，包括模擬老化、依良率差別分等級、建立有效資料供分析使用等。因IC製程微縮速度愈加快速，整合及複雜度漸高，測試業除了生產管理、設備、測試、品保、軟硬體等各類別工程師之外，各公司為了提高競爭力，漸漸投入技術研發，對研發工程師需求也增多。請繼續往下閱讀...台灣測試廠至少30家台灣測試廠至少30家台灣測試業至少超過30家，知名上市上櫃公司包括京元電、力成、泰林、矽格、欣銓、久元等，另有在海外掛牌的南茂、聯測等公司。各家公司測試產品別互有差異，有的以記憶體為主，代表性公司如力成、泰林等，京元電的邏輯產品比重較高，欣銓以12吋晶圓測試為主，日月光、矽品則以封裝為主。  業者表示，國際全球IDM（整合元件）大廠為了降低營運成本，近幾年開始，陸續釋出後段封測訂單委外代工，這使得台灣測試業營收明顯上揚，不過，隨著技術提高，測試機每台達數千萬元或上億元，測試業已成為資本密集化，同時也需透過與國際大廠策略聯盟，才有競爭力可言，這顯示測試業將走向「大者恆大」局面。應試基本條件︰電學、程式、英文應試基本條件︰電學、程式、英文起薪行情3萬2到3萬6起薪行情3萬2到3萬6記者洪友芳／專題報導工程師為測試業核心專業人力，需求人才以電子、電機、材料等理工相關科系畢業者為主，大學畢業無經驗的新人，起薪行情約3萬2千元到3萬6千元之間，依科系別及個人潛力而定；研究所畢業起薪標準約在3萬9千元上下。1位測試大廠主管表示，測試業工程師至少需要具備基本電學、軟體程式及英文能力，因設備機台是測試業最大的資本支出，也都以英文顯現，客戶有不少比例為國外廠商，工程師需懂電學、程式及英文，才能勝任測試業工作。此外，這位主管也說，測試業沒有自有產品，屬於專業服務業，以測試客戶產品居多，工程師經常需要跟客戶接洽、溝通，滿足客戶需求，工程師的服務熱忱、待人處事態度也要能屈能伸，公司的在職進修於安排專業課程之外，也會有人際溝通等強化客戶服務課程。下班時間常不固定下班時間常不固定京元電人力資源處處長柯芳蓉指出，測試業所需的專業人力，部分是跟前段的IC設計、生產製造相互競爭，基本上，前段的待遇稍高，工作性質也有別，但有的求職者偏好後段測試業，認為發展空間較大。柯芳蓉表示，測試業的工程師以常態班為主，上班時間通常是上午8點多到5點半或6點，但由於半導體業都屬責任制，測試業工程師的下班時間其實並不固定，有時因應客戶交貨趕時間，延到8、9點或半夜下班也是常有的事。論工作甘苦，辛苦是半導體業的共通處，但為客戶解決問題、令客戶滿意時，卻讓人很有工作成就感。封測類股昨於盤中紛跌停，評價： '''
# lm_response(content)

In [9]:
def calculate_return(signal, threshold):
    # signal = [日期, 評分, 價格]
    hold = 0  # 持有的股數
    balance = 10000000
    init_balance = balance

    for item in signal:
        score = float(item[1])
        price = float(item[2])

        if hold == 0:
            if score == 2:
                hold += int(balance/price)
                balance = 0
            elif score == 1:
                hold += int((balance/2)/price)
                balance = int(balance/2)
        elif hold > 0:
            if score < 0:
                balance += price*hold
                hold = 0
            elif score == 2:
                hold += int(balance/price)
                balance = 0
    
    balance += hold*price
    last_price = float(signal[len(signal) - 1][2])
    bnh = last_price / float(signal[0][2])
    r = balance/init_balance

    return round(r, 5), round(bnh, 5)

In [10]:
def write_output(content, raw_data, folder_path):
    # 輸出結果
    count = 1
    while True:
        if os.path.exists(folder_path + str(count)):
            count += 1
            continue
        else:
            os.makedirs(folder_path + str(count))
            with open(folder_path + str(count) + "/detail.json" , "w", encoding="UTF-8") as f:
                f.write(content)
            
            for i in range(len(raw_data)):
                #寫入
                file = folder_path + str(count) + "/" + str(i+1) + ".csv"
                pd.DataFrame(raw_data[i]).to_csv(file)
                # with open(folder_path + str(count) + "/" + str(i+1), "w", encoding="UTF-8") as f:
                #     f.write(pd.DataFrame(raw_data[i]))
            print("寫入" + folder_path + str(count))
            break

In [11]:
# # 提示變異
# def mutate_prompt(p_mutate):
#     llm = ChatOllama(model=model, max_tokens=128, temperature=temperature, top_p=top_p)
#     system_prompt = '你是一個厲害的語言助理，會完成使用者要求並用繁體中文回答'
    
#     prompt =  f'''<|begin_of_text|><|start_header_id|>system<|end_header_id|> 

#     {system_prompt}<|eot_id|> 
#     <|start_header_id|>user<|end_header_id|>

#     請回答以下句子的1個變異，同時保留語意："{p_mutate}"<|eot_id|>
#     <|start_header_id|>assistant<|end_header_id|>'''
    
#     response = llm.invoke(prompt)
#     return response

# out = mutate_prompt('對業務創新計劃進行詳盡的審查，專注於預計對目標市場的影響，並探索投資者樂觀的長期增長展望刺激的股票價格上漲趨勢')
# print(out.content)

# 主程式

In [12]:
run_count = 10

start_time = datetime(2023, 1, 1)
end_time = datetime(2023, 12, 31)

stock_ids = ["1101", "2211", "2385", "2542", "2880", "2912", "3023", "3264", "5269", "8027"]
stock_names = ["台泥", "長榮鋼", "群光", "興富發", "華南金", "統一超", "信邦", "欣銓", "祥碩", "鈦昇"]
# stock_ids = ["2912", "3023", "3264", "5269", "8027"] # 執行時間:108 min
# stock_names = ["統一超", "信邦", "欣銓", "祥碩", "鈦昇"]
# stock_ids = ["8027"] # 執行時間：4 min
# stock_names = ["鈦昇"]

p_mutate = '業務創新計劃進行詳盡的審查，專注於預計對目標市場的影響，並探索投資者樂觀的長期增長展望刺激的股票價格上漲趨勢'
p_mutate_2 = p_mutate
rtn_ = 0

for i in range(run_count):
    rtns = []
    bnhs = []
    raw_data = []
    for idx, stock_id in enumerate(stock_ids):
        print(f'變異{i + 1}次\t執行{stock_id} {stock_names[idx]}')
        qurey = f'請依據"{p_mutate}"對（{stock_id}）{stock_names[idx]}的影響判斷'
        folder_path = "stock_news/" + stock_id + "/" + stock_id + "_news/"
        files = os.listdir(folder_path)
        sorted_files = sorted(files)

        # 載入買進價格
        df = pd.read_csv("price_history/" + stock_id + ".csv", dtype=str)
        price_np = df.to_numpy()
        prices = price_np.tolist()  # [時間,價格]

        signal_2 = []

        # 清空檔案
        with open("concat_news", "w") as f:
            f.write("")

        for price in prices:
            # 檢查日期是否在參考日期之後
            try:
                date_obj = datetime.strptime(price[0], '%Y%m%d')
            except:
                continue

            if not (start_time <= date_obj <= end_time):
                continue
            
            sig = 0
            for index, file_name in enumerate(sorted_files):
                if file_name == price[0]:
                    with open("concat_news", "a") as f:
                        with open(folder_path+file_name, "r") as f2:
                            f.write(f'{file_name}日新聞：{f2.read()}\n')
                    sig = lm_response("concat_news", qurey)    # file_name, qurey
                    break
            
            signal_2.append([price[0], sig, price[1]]) # [日期, 評分, 價格]
            print(f'{price[0]}: {sig}')

        signal_2_np = np.array(signal_2)
        # 回傳 IRR 以及總額投入
        rtn, bnh = calculate_return(signal_2_np, 0)
        rtns.append(rtn)
        bnhs.append(bnh)
        raw_data.append(signal_2_np.tolist())
        # print("rtn", irr)
        # print("總額投入報酬率", bnh)


    # 輸出結果
    output_data = {
        "date_start": start_time.strftime("%Y%m%d"),
        "date_end": end_time.strftime("%Y%m%d"),
        "info": []
    }

    for j in range(len(stock_ids)):
        stock_info = {
            "stock_id": stock_ids[j],
            "rtn": rtns[j],
            "bnh": bnhs[j]
        }
        output_data["info"].append(stock_info)

    output_data["total_rtn"] = sum(rtns) / len(rtns)
    output_data["total_bnh"] = sum(bnhs) / len(bnhs)
    output_data["model"] = model
    output_data["temperature"] = temperature
    output_data["top_p"] = top_p
    output_data["seed"] = seed
    output_data["system_prompt"] = system_prompt
    output_data["prompt_mutate"] = p_mutate_2
    output_data["chunk_size"] = chunk_size
    output_data["chunk_overlap"] = chunk_overlap


    # 轉換成 JSON 格式
    json_content = json.dumps(output_data, ensure_ascii=False, indent=4)

    write_folder = "eval_llama/output"
    write_output(content=json_content, raw_data=raw_data, folder_path=write_folder)

    # 判斷是否更改變異
    # if sum(rtns) / len(rtns) > rtn_:
    #     print("更改變異提示為:", p_mutate_2)
    #     p_mutate_2 = mutate_prompt(p_mutate)
    #     rtn_ = sum(rtns) / len(rtns)

變異1次	執行1101 台泥


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

<INST>非常好</INST>

As a seasoned stock analyst, I must say that the situation is very favorable for Taiwan Cement (台泥). With the government's capital injection and support for the "安心股" (stable stocks), there are high expectations for Taiwan Cement to perform well in the long term. Additionally, the news of foreign funds pouring into the market has further boosted the stock's prospects.

The article highlights that Taiwan Cement is one of the 15 "安心股" with a high chance of being included as a key indicator for the new year. This implies that the government and institutional investors are confident in the company's performance, which can drive up its stock price. Furthermore, foreign funds have been net buyers of Taiwan Cement shares, indicating increased interest and confidence in the company.

Given these factors, I believe that Taiwan Cement is likely to perform well and show a strong upward trend in its stock price. Therefore, my answer is:</INST>非常好</INST>
20230103: 2


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

🤔 As a seasoned stock analyst, I'd say that the recent announcement by Ta-Ning (1101) regarding its employee shareholding plan is a positive development. 📈

The plan aims to allow employees to participate in the company's equity through share purchases, which can help boost morale and motivation. 👥 This, in turn, can lead to improved productivity and competitiveness, ultimately benefiting the company's long-term growth prospects. 🔥

Moreover, the plan's structure, which sets a limit on the number of shares each employee can purchase, demonstrates a commitment to fairness and equity among employees. 💰 This is likely to increase trust and confidence in the company's management, further enhancing its reputation and appeal to investors. 📊

In terms of market impact, I believe that this announcement has the potential to stimulate long-term growth expectations and drive up stock prices. 💸 After all, when employees are more invested (pun intended) in a company's success, they're more likely t

...

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

[INST] #非常好 [/INST]

（根據文章，台泥以年產能5,633萬公噸位居第七，這表示台泥在大陸水泥生產業界有很高的競爭力和穩定性。如果我們認為大陸水泥需求將繼續增加，那麼台泥的長期增長展望是非常好。但是，需要注意的是這只是一個簡單的分析結果，並且還需要考慮其他因素，例如公司治理、財務狀況等。[/INST]
20230411: 2


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

[INST]非常好 [/INST]

As a stock analyst, I would say that the news is very positive for Taiwan Cement (1101). Being ranked 7th in terms of cement production capacity in China for 2022 is a significant achievement. This not only demonstrates the company's ability to maintain its market share but also implies that it has been able to increase its production capacity, which can lead to increased revenue and profitability.

Additionally, being outranked by other major players such as Anhui Conch (1102) at number 10 does not necessarily mean that Taiwan Cement is performing poorly. Instead, it highlights the competitive nature of the cement industry in China, where many players are vying for market share.

Overall, this news suggests that Taiwan Cement has been able to maintain its competitiveness and even gain ground in a highly competitive market. This can lead to increased investor confidence and a potential increase in stock price over the long term.
20230412: 2


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]